In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import utilities.plot_settings
from matplotlib import rc
from matplotlib import cm
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde, ks_2samp
from matplotlib.lines import Line2D
from matplotlib.ticker import LogFormatterMathtext
from scipy.stats import gaussian_kde
from matplotlib.patches import Patch
import astropy.constants as astro_cons
from mlpoppyns.simulator.config_simulator import cfg
from pathlib import Path

rc("text", usetex=True)
rc("font", family="serif")
plt.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"
yellow = plt.cm.Set2(5 / 7)
orange = plt.cm.Set2(1 / 7)
sea = plt.cm.Set2(0 / 7)
blue = plt.cm.Set2(2 / 7)
green = plt.cm.Set2(4 / 7)
purple = plt.cm.Set2(3 / 7)


In [ ]:
SMALL_SIZE = 12
MEDIUM_SIZE = 20
BIGGER_SIZE = 22

plt.rc("font", size=BIGGER_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=BIGGER_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=BIGGER_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=BIGGER_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=BIGGER_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=MEDIUM_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title


# Check Simulations

## Load X-ray data

In [ ]:
# Read the thermally emitting neutron star catalog.csv file. Binary pulsars are excluded.
df_x_obs = pd.read_csv(
    "/Users/celsapardoaraujo/Documents/ML-Poppyns/data/observations/thermal_NS_05-11-2024.csv",
    delimiter=",",
    header=[0],
)
# We remove Central Compact Objects (CCOs) because their magnetic field have likely been buried
# due to supernova fallback and their evolution cannot be modelled with our simulation framework.
# We also remove neutron stars associated with pulsar wind nebulae as their detection might have been triggered by the non-thermal emission of the nebula.
# We also remove those stars that belong to the Magellanic Clouds.
df_x_obs = df_x_obs[~df_x_obs["class"].isin(["CCO", "RPP"])]
df_x_obs = df_x_obs[~df_x_obs["assoc"].isin(["PWN", "PWN, Radio"])]
df_x_obs = df_x_obs[~df_x_obs["assoc"].isin(["SMC", "LMC"])]
df_x_obs = df_x_obs.dropna(subset=["period(s)"])
# df_x_obs = df_x_obs.dropna(subset=["pdot(1e-11s/s)"])
df_x_obs = df_x_obs.dropna(subset=["abs. flux (0.3-10 keV)"])

### Load Magnetars

In [ ]:
magnetar_sources_in_obs = ['1E1841-045',  'PSRJ1119-6127', 'CXOUJ171405.7-3810',
       'PSRJ1622-4950', 'SGR1806-20', 'SGRJ1745-2900', 'SGR1900+14',
       'SwiftJ1818.0-1607', '1E1547-5408']# We exclude 'PSRJ1846-0258' because its emission is from the PWN
df_x_obs_young = df_x_obs[df_x_obs["source"].isin(magnetar_sources_in_obs)]
df_x_obs_old = df_x_obs[~df_x_obs["source"].isin(magnetar_sources_in_obs)]
df_x_obs_old = df_x_obs_old[~df_x_obs_old["class"].isin(["XDINS"])]
# Step 2: filter your main dataset for these sources

S_x_abs_obs_young = df_x_obs_young["abs. flux (0.3-10 keV)"].to_numpy().astype(np.float64)
P_x_obs_young = df_x_obs_young["period(s)"].to_numpy().astype(np.float64)
Pdot_x_obs_young = df_x_obs_young["pdot(1e-11s/s)"].to_numpy().astype(np.float64) * 1e-11

S_x_abs_obs = df_x_obs_old["abs. flux (0.3-10 keV)"].to_numpy().astype(np.float64)
P_x_obs = df_x_obs_old["period(s)"].to_numpy().astype(np.float64)
Pdot_x_obs = df_x_obs_old["pdot(1e-11s/s)"].to_numpy().astype(np.float64) * 1e-11

df_x_obs_young

### Load XDINS

In [ ]:
xdins_sources_in_obs = ['RXJ0420.0-5022', 'RXJ1856.5-3754', 'RXJ2143.0+0654', 'RXJ0720.4-3125',
       'RXJ0806.4-4123', 'RXJ1308.6+2127']

# Step 2: filter your main dataset for these sources
df_x_obs_xdins = df_x_obs.loc[
    df_x_obs['source'].isin(xdins_sources_in_obs)
]
S_x_abs_obs_xdins = df_x_obs_xdins["abs. flux (0.3-10 keV)"].to_numpy().astype(np.float64)
P_x_obs_xdins = df_x_obs_xdins["period(s)"].to_numpy().astype(np.float64)
Pdot_x_obs_xdins = df_x_obs_xdins["pdot(1e-11s/s)"].to_numpy().astype(np.float64) * 1e-11
dist_x_obs = df_x_obs_xdins["distance(kpc)"].to_numpy().astype(np.float64)
L_x_obs = df_x_obs_xdins["luminosity_bol_qui(e33erg/s)"].to_numpy().astype(np.float64)

In [ ]:
data_psr = pd.read_csv('/Users/celsapardoaraujo/Documents/data/magnetar-br/final_snr_psr_assoc_dataset.csv')
data_psr.columns

In [ ]:
mask_data_young_2kyrs = ((data_psr['age_min (yr)'] <= 2000) | (data_psr['tau_c_psr[kyrs]'] < 2)) & (data_psr['ns_name'].notna())
remove_list = ["J1640-4631", "J1622-4950", "J1721-3532", 
               "J1550-5418", "J1907+0919"]

df_filtered = data_psr[~data_psr["ns_name"].isin(remove_list)]
data_table = df_filtered[mask_data_young_2kyrs][['snr_name','ns_name', 'Period[s]',"Pdot[s/s]",'B_dip_1e14G_at_pole','Erot' ,'class']]

In [ ]:
data_table.loc[data_table['ns_name'] == '1E1547-5408', 'class'] = 'Magnetar'

# 2) Build a class column for plotting: NaN -> RPP, except the magnetar we just fixed
data_table['class_plot'] = data_table['class'].fillna('RPP')


In [ ]:
data_table[data_table['class_plot'] =='RPP']

In [ ]:
psr_2kyrs = ['J1747-2809','J1813-1749','J1833-1034','J1930+1852','J0205+6449','J0534+2200','J1400-6325','PSRB1509-58']

In [ ]:
rpp_obs = data_psr[data_psr["ns_name"].isin(psr_2kyrs)][[ 'ns_name', 'Period[s]', 'Pdot[s/s]']]
P_rpp_obs = rpp_obs['Period[s]']
Pdot_rpp_obs = rpp_obs['Pdot[s/s]']

In [ ]:
rpp_obs

In [ ]:
light_c_cgs = astro_cons.c.cgs.value
solar_mass_cgs = astro_cons.M_sun.cgs.value

NS_mass = cfg['NS_mass']
NS_radius = cfg['NS_radius']
I_ns = (2/5) * NS_mass * (NS_radius**2)

def rotational_energy(P,Pdot):
    Edot_ns = (4*np.pi**2) * I_ns * Pdot / (P**3)
    return Edot_ns

def Dipolar_B_at_pole(P,Pdot):
    Bdip_eq_ns = np.sqrt(((3 * light_c_cgs**3) * I_ns * P * Pdot ) / (8 * np.pi**2 * NS_radius**6 ))
    Bdip_p_ns = 2 * Bdip_eq_ns/1.e14
    return Bdip_p_ns



# Load RPP from ATNF Catalog


In [ ]:
# Read the full ATNF catalog.csv file. Binary pulsars are excluded.
df_atnf = pd.read_csv(
    "/Users/celsapardoaraujo/Documents/ML-Poppyns/data/observations/atnf_full_nobinary_25-03-2025_with_errors.csv",
    delimiter=";",
    header=[0, 1],
)
df_atnf.columns

df_atnf.columns = df_atnf.columns.droplevel(1)

# Select only stars that are not in globular clusters or in the Magellanic Clouds.
discard = [
    "EXGAL:SMC",
    "EXGAL:LMC",
    "GC:47Tuc",
    "GC:M3",
    "GC:M5",
    "GC:M13",
    "GC:NGC6440",
    "GC:Ter5",
    "GC:NGC6441",
    "GC:NGC6517",
    "GC:NGC6522",
    "GC:NGC6624",
    "GC:M28(NGC6626)",
    "GC:NGC6652",
    "GC:M22(NGC6656)",
    "GC:NGC6752",
    "GC:NGC6760",
    "GC:M15",
    "GC:M30",
]

df_atnf = df_atnf[~df_atnf["ASSOC"].str.match("|".join(discard))]

# Select only isolated non-recycled neutron stars through filters with P > 0.01 and Pdot > 1e-19 (for those with measured values).
df_atnf = df_atnf[df_atnf["P0"].to_numpy().astype(np.float64) > 0.01]

df_atnf = df_atnf[
    (df_atnf["P1"].to_numpy().astype(np.float64) > 1.0e-19)
    | (df_atnf["P1"].isin(["NAN"]))
]
df_atnf = df_atnf[df_atnf["P1"].astype(float) < 1e-10]

P_obs = df_atnf["P0"].to_numpy().astype(np.float64)
Pdot_obs = df_atnf["P1"].to_numpy().astype(np.float64)

In [ ]:
def sigma_levels(Z, sigmas=[1, 2, 3]):
    """
    Given a 2D density grid Z (not log), return contour levels for
    1σ, 2σ, 3σ corresponding to enclosed probability.
    """
    # target cumulative probabilities
    probs = [0.6827, 0.9545, 0.9973]
    
    # normalize
    Z = Z - Z.min()
    Z = Z / Z.sum()
    
    # sort from highest density to lowest
    Z_flat = Z.ravel()
    idx = np.argsort(Z_flat)[::-1]
    Z_sorted = Z_flat[idx]
    cumsum = np.cumsum(Z_sorted)
    
    # find density threshold for each probability
    levels = []
    for p in probs:
        cutoff = Z_sorted[np.searchsorted(cumsum, p)]
        levels.append(cutoff)

    return levels


In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogFormatterMathtext
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde

# -------------------------------------------------------------------
# Assumes these arrays already exist in your session:
#   P_x_obs, Pdot_x_obs
#   P_x_obs_young, Pdot_x_obs_young
#   P_rpp_obs, Pdot_rpp_obs
#   P_obs, Pdot_obs
#   P_x_obs_xdins, Pdot_x_obs_xdins, S_x_abs_obs_xdins
#   S_x_abs_obs_young
# -------------------------------------------------------------------

# -------------------------------------------------------------------
# Paths & basic config
# -------------------------------------------------------------------
# Magnetar simulations
parent_path = pathlib.Path(
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/"
    "simulations_sim_vs_obs_figure/magnetar_rpp"
)
exp_folders = [
    "simulations_7_5e13",
    "simulations_1e14",
    "simulations_2_5e14",
    "simulations_5e14",
    "simulations_7_5e14",
    "simulations_1e15",
]
n_simulations = 25

# XDINS simulations (to overlay)
parent_path_xdins = pathlib.Path(
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/"
    "simulations_sim_vs_obs_figure/xdins"
)
xdins_exp_folders = [
    "exp_7_5e13",
    "exp_1e14",
    "exp_2_5e14",
    "exp_5e14",
    "exp_7_5e14",
    "exp_1e15",
]

NS_inertia = 2.0 / 5.0 * 1.4 * 2.0e33 * (1.259e6 ** 2)

# Subtitles
B0_labels = [
    r"$B_{0,\rm mag} = 7.5 \times 10^{13}\ \rm G$",
    r"$B_{0,\rm mag} = 10^{14}\ \rm G$",
    r"$B_{0,\rm mag} = 2.5 \times 10^{14}\ \rm G$",
    r"$B_{0,\rm mag} = 5 \times 10^{14}\ \rm G$",
    r"$B_{0,\rm mag} = 7.5 \times 10^{14}\ \rm G$",
    r"$B_{0,\rm mag} = 10^{15}\ \rm G$",
]

# -------------------------------------------------------------------
# Prepare figure
# -------------------------------------------------------------------
fig, axes = plt.subplots(
    2, len(exp_folders),
    figsize=(16, 10),
    sharex=True,
    constrained_layout=True
)

xdins_present = False  # to know if we should add XDINS to simulated legend

for j, exp_name in enumerate(exp_folders):
    # -----------------------------
    # Load magnetar simulations
    # -----------------------------
    base_path = parent_path / exp_name

    S_x_rcs_abs_list, P_x_sim_list, Pdot_x_sim_list = [], [], []

    for i in range(1, n_simulations):
        sim_path = base_path / f"simulation_{i}" / "final_population.pkl.gz"
        data_pop = pd.read_pickle(sim_path, compression="gzip")

        magnetars = data_pop[
            (np.log10(data_pop[('B', '[G]')]) >= 13.5) &
            (data_pop[('S_x_rcs_abs', '[erg s^-1 cm^-2]')] > 1e-14)
        ]

        S_x_rcs_abs_list.append(
            magnetars[('S_x_rcs_abs', '[erg s^-1 cm^-2]')].to_numpy()
        )
        P_x_sim_list.append(magnetars[('P', '[s]')].to_numpy())
        Pdot_x_sim_list.append(magnetars[('P_dot', '[s s^-1]')].to_numpy())

    P_all = np.concatenate(P_x_sim_list)
    Pdot_all = np.concatenate(Pdot_x_sim_list)
    S_all = np.concatenate(S_x_rcs_abs_list)

    # -----------------------------
    # Load best-simulated RPPs
    # -----------------------------
    data_rpp = pd.read_pickle(
        "/Users/celsapardoaraujo/Documents/ML-Poppyns/data/paper_results/"
        "pardo_etal_2025/best_inferred_atnf/"
        "survey_HTRU_low_mid_results.pkl.gz",
        compression="gzip"
    )
    P_rpp_all = data_rpp[('P', '[s]')].to_numpy()
    Pdot_rpp_all = data_rpp[('P_dot', '[s s^-1]')].to_numpy()

    # -----------------------------
    # Load XDINS sims (if available for this column)
    # -----------------------------
    P_xdins_all = None
    Pdot_xdins_all = None
    S_xdins_all = None

    if j < len(xdins_exp_folders):
        base_path_x = parent_path_xdins / xdins_exp_folders[j]

        xdins_P_list = []
        xdins_Pdot_list = []
        xdins_S_list = []

        n_simulations_xdins = 10

        for i in range(1, n_simulations_xdins):
            sim_path_x = base_path_x / f"simulation_{i}" / "final_population.pkl.gz"
            data_pop_x = pd.read_pickle(sim_path_x, compression="gzip")

            r = data_pop_x["r"]["[kpc]"].to_numpy()
            phi = data_pop_x["phi"]["[rad]"].to_numpy()
            z = data_pop_x["z"]["[kpc]"].to_numpy()

            x = r * np.cos(phi)
            y = r * np.sin(phi)
            dist_from_sun = np.sqrt(
                (x - 0.0) ** 2 +
                (y - 8.3) ** 2 +
                (z - 0.006) ** 2
            )

            mask_dist = dist_from_sun <= 0.5
            mask_flux = data_pop_x[('S_x_rcs_abs', '[erg s^-1 cm^-2]')] > 5e-13

            data_pop_x_sel = data_pop_x[mask_dist & mask_flux]
            mask_age_real = data_pop_x_sel["age"]["[yr]"] > 1e5
            xdins = data_pop_x_sel[mask_age_real]

            P_x_sim = xdins[('P', '[s]')].to_numpy()
            Pdot_x_sim = xdins[('P_dot', '[s s^-1]')].to_numpy()
            S_x_rcs_abs = xdins[('S_x_rcs_abs', '[erg s^-1 cm^-2]')].to_numpy()

            if len(P_x_sim) > 0:
                xdins_P_list.append(P_x_sim)
                xdins_Pdot_list.append(Pdot_x_sim)
                xdins_S_list.append(S_x_rcs_abs)

        if len(xdins_P_list) > 0:
            P_xdins_all = np.concatenate(xdins_P_list)
            Pdot_xdins_all = np.concatenate(xdins_Pdot_list)
            S_xdins_all = np.concatenate(xdins_S_list)
            xdins_present = True

    # ----------------------------------------------------------------
    # 1st row: P vs Pdot
    # ----------------------------------------------------------------
    ax1 = axes[0, j]

    # Grid limits including magnetars, RPPs, and XDINS (if present)
    P_min = min(P_all.min(), P_rpp_all.min())
    P_max = max(P_all.max(), P_rpp_all.max())
    Pdot_min = min(Pdot_all.min(), Pdot_rpp_all.min())
    Pdot_max = max(Pdot_all.max(), Pdot_rpp_all.max())

    if P_xdins_all is not None:
        P_min = min(P_min, P_xdins_all.min())
        P_max = max(P_max, P_xdins_all.max())
    if Pdot_xdins_all is not None:
        Pdot_min = min(Pdot_min, Pdot_xdins_all.min())
        Pdot_max = max(Pdot_max, Pdot_xdins_all.max())

    x_min, x_max = np.log10(P_min), np.log10(P_max)
    y_min, y_max = np.log10(Pdot_min), np.log10(Pdot_max)

    X, Y = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )
    positions = np.vstack([X.ravel(), Y.ravel()])

    # KDE for simulated magnetars
    xy_mag = np.vstack([np.log10(P_all), np.log10(Pdot_all)])
    kde_mag = gaussian_kde(xy_mag)
    Z_mag = np.reshape(kde_mag(positions), X.shape)

    # KDE for simulated RPPs (if needed later)
    xy_rpp = np.vstack([np.log10(P_rpp_all), np.log10(Pdot_rpp_all)])
    kde_rpp = gaussian_kde(xy_rpp)
    Z_rpp = np.reshape(kde_rpp(positions), X.shape)

    # KDE for simulated XDINS (if present)
    if P_xdins_all is not None:
        xy_xdins = np.vstack([np.log10(P_xdins_all), np.log10(Pdot_xdins_all)])
        kde_xdins = gaussian_kde(xy_xdins)
        Z_xdins = np.reshape(kde_xdins(positions), X.shape)
    else:
        Z_xdins = None

    # Observed magnetars & RPPs
    ax1.plot(
        P_x_obs, Pdot_x_obs,
        linestyle="None",
        marker="o",
        color="tab:purple",
        alpha=0.3,
        markersize=10,
        rasterized=True,
        label="Obs Mag" if j == 0 else None
    )
    ax1.plot(
        P_x_obs_young, Pdot_x_obs_young,
        linestyle="None",
        marker="X",
        color="tab:purple",
        markersize=10,
        rasterized=True,
        label="Obs Young Mag" if j == 0 else None
    )
    ax1.plot(
        P_rpp_obs, Pdot_rpp_obs,
        linestyle="None",
        marker="*",
        color="grey",
        markersize=7,
        rasterized=True,
        label="Obs Young RPPs" if j == 0 else None
    )
    ax1.plot(
        P_obs, Pdot_obs,
        linestyle="None",
        marker="o",
        color="grey",
        markersize=7,
        alpha=0.1,
        rasterized=True,
        label="Obs RPPs" if j == 0 else None
    )

    # Contours: Simulated magnetars
    ax1.contour(
        10 ** X, 10 ** Y, Z_mag,
        levels=[0.1, 0.3, 0.7, 0.9, 1.1, 1.3],
        colors="tab:purple",
        linestyles="dashed",
        alpha=0.7
    )

    # Contours: Simulated XDINS + observed XDINS
    if Z_xdins is not None:
        ax1.contour(
            10 ** X, 10 ** Y, Z_xdins,
            levels=5,
            colors="tab:orange",
            linestyles="dashed",
            alpha=0.6
        )
        ax1.plot(
            P_x_obs_xdins, Pdot_x_obs_xdins,
            linestyle="None",
            marker="X",
            color="tab:orange",
            markersize=10,
            rasterized=True,
            label="Obs XDINS" if j == 0 else None
        )

    # Axes formatting
    ax1.set_xscale("log")
    ax1.set_yscale("log")
    ax1.set_xlim(1.0e-1, 100.0)
    ax1.set_ylim(1e-17, 1.0e-9)

    if j == 0:
        ax1.set_ylabel(r"$\dot{P}$ [s/s]")
    else:
        ax1.set_ylabel("")
        ax1.set_yticklabels([])
        ax1.set_yticks([])

    ax1.set_title(B0_labels[j], fontsize = 20)
    ax1.yaxis.set_major_formatter(LogFormatterMathtext())
    ax1.xaxis.set_major_formatter(LogFormatterMathtext())
    ax1.tick_params(axis='both', which='major')
    ax1.tick_params(axis='both', which='minor')

    # ----------------------------------------------------------------
    # 2nd row: P vs S_x_rcs_abs
    # ----------------------------------------------------------------
    ax2 = axes[1, j]

    xy = np.vstack([np.log10(P_all), np.log10(S_all)])
    kde = gaussian_kde(xy)

    # Include XDINS in limits if present
    xmin = np.log10(
        min(P_all.min(), P_xdins_all.min() if P_xdins_all is not None else P_all.min())
    )
    xmax = np.log10(
        max(P_all.max(), P_xdins_all.max() if P_xdins_all is not None else P_all.max())
    )
    ymin = np.log10(
        min(S_all.min(), S_xdins_all.min() if S_xdins_all is not None else S_all.min())
    )
    ymax = np.log10(
        max(S_all.max(), S_xdins_all.max() if S_xdins_all is not None else S_all.max())
    )

    X2, Y2 = np.meshgrid(
        np.linspace(xmin, xmax, 200),
        np.linspace(ymin, ymax, 200)
    )
    positions2 = np.vstack([X2.ravel(), Y2.ravel()])
    Z = np.reshape(kde(positions2), X2.shape)

    # Magnetar contours
    ax2.contour(
        10 ** X2, 10 ** Y2, Z,
        levels=5,
        colors="tab:purple",
        linestyles="dashed",
        alpha=0.6
    )

    # XDINS contours + points (orange)
    if S_xdins_all is not None:
        xy_xdins2 = np.vstack([np.log10(P_xdins_all), np.log10(S_xdins_all)])
        kde_xdins2 = gaussian_kde(xy_xdins2)
        Z_xdins2 = np.reshape(kde_xdins2(positions2), X2.shape)

        ax2.contour(
            10 ** X2, 10 ** Y2, Z_xdins2,
            levels=5,
            colors="tab:orange",
            linestyles="dashed",
            alpha=0.6
        )

        ax2.plot(
            P_x_obs_xdins, S_x_abs_obs_xdins,
            linestyle="None",
            marker="X",
            color="tab:orange",
            markersize=10,
            rasterized=True
        )

    ax2.plot(
        P_x_obs_young, S_x_abs_obs_young,
        linestyle="None",
        marker="X",
        color="tab:purple",
        rasterized=True,
        markersize=10,
        label="Obs Mag" if j == 0 else None
    )

    ax2.set_xscale("log")
    ax2.set_yscale("log")
    ax2.set_xlim(5e-3, 300.0)
    ax2.set_ylim(1.0e-14, 1.0e-9)
    ax2.set_xticks([1e-1, 1e0, 1e1, 1e2])
    ax2.set_xlabel("P [s]")

    if j == 0:
        ax2.set_ylabel(r"$S_{X,\rm abs}$ [erg s$^{-1}$ cm$^{-2}$]")
    else:
        ax2.set_ylabel("")
        ax2.set_yticklabels([])
        ax2.set_yticks([])

    ax2.yaxis.set_major_formatter(LogFormatterMathtext())
    ax2.xaxis.set_major_formatter(LogFormatterMathtext())
    ax2.tick_params(axis='both', which='major')
    ax2.minorticks_off()

# -------------------------------------------------------------------
# SINGLE LEGEND ON THE RIGHT (NO TITLES)
# -------------------------------------------------------------------

# Get all observed handles from the first top-left panel
obs_handles, obs_labels = axes[0, 0].get_legend_handles_labels()

# Proxy artists for simulated populations
sim_mag_proxy = Line2D([0], [0], color="tab:purple", linestyle="dashed")
sim_xdins_proxy = Line2D([0], [0], color="tab:orange", linestyle="dashed")

sim_handles = [sim_mag_proxy]
sim_labels = ["Sim Mag"]

if xdins_present:
    sim_handles.append(sim_xdins_proxy)
    sim_labels.append("Sim XDINS")

# Combine everything into ONE legend
all_handles = obs_handles + sim_handles
all_labels = obs_labels + sim_labels

fig.legend(
    all_handles,
    all_labels,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),   # RIGHT side, vertically centered
    frameon=True
)
# -------------------------------------------------------------------
# Final layout and save
# -------------------------------------------------------------------
plt.subplots_adjust(hspace=0, wspace=0)
plt.savefig(
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/plots/"
    "plot_pop_syn_ppdot_with_xdins_.pdf",
    bbox_inches="tight"
)
plt.show()


# Cumulative distribution of Period, Pdot and Xray fluxes Magnetars

In [ ]:

# Yellow → Green gradient for increasing B0
cmap = cm.get_cmap("YlGn")
colors = cmap(np.linspace(0.2, 0.9, 12))
parent_path = pathlib.Path("/Users/celsapardoaraujo/Documents/data/magnetar-br/simulations_sim_vs_obs_figure/magnetar_rpp")
exp_folders = ["simulations_7_5e13",
               "simulations_1e14",
               "simulations_1_5e14",
               "simulations_2e14",
               "simulations_2_5e14",
              "simulations_5e14",
              "simulations_7_5e14",
              "simulations_1e15"]
n_simulations = 25

# Subtitles
B0_labels = [
    r"$B_{0,\rm mag} = 7.5 \times 10^{13}\ \rm G$",
    r"$B_{0,\rm mag} = 10^{14}\ \rm G$",
    r"$B_{0,\rm mag} = 1.5 \times 10^{14}\ \rm G$",
    r"$B_{0,\rm mag} = 2 \times 10^{14}\ \rm G$",
    r"$B_{0,\rm mag} = 2.5 \times 10^{14}\ \rm G$",
    r"$B_{0,\rm mag} = 5 \times 10^{14}\ \rm G$",
    r"$B_{0,\rm mag} = 7.5 \times 10^{14}\ \rm G$",
    r"$B_{0,\rm mag} = 10^{15}\ \rm G$",
]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plt.subplots_adjust(wspace=0)   # smaller = less horizontal space
axP, axPdot, axS = axes


# ============================
#      LOOP OVER SIMULATIONS
# ============================
# ============================
#      LOOP OVER SIMULATIONS
# ============================
for j, exp_name in enumerate(exp_folders):
    base_path = parent_path / exp_name
    
    S_x_rcs_abs_list, P_x_sim_list, Pdot_x_sim_list = [], [], []
    
    # --- find all existing simulation_* folders ---
    sim_dirs = sorted(
        [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith("simulation_")],
        key=lambda p: int(p.name.split("_")[1])  # sort by the number after "simulation_"
    )

    for sim_dir in sim_dirs:
        sim_path = sim_dir / "final_population.pkl.gz"
        if not sim_path.exists():
            # skip if the file is missing, just in case
            continue
        
        data_pop = pd.read_pickle(sim_path, compression="gzip")
        
        magnetars = data_pop[
            (np.log10(data_pop[('B','[G]')]) >= 13.5) &
            (data_pop[('S_x_rcs_abs','[erg s^-1 cm^-2]')] > 1e-14)
        ]
        
        S_x_rcs_abs_list.append(
            magnetars[('S_x_rcs_abs','[erg s^-1 cm^-2]')].to_numpy()
        )
        P_x_sim_list.append(
            magnetars[('P','[s]')].to_numpy()
        )
        Pdot_x_sim_list.append(
            magnetars[('P_dot','[s s^-1]')].to_numpy()
        )

    # --- Merge ---
    P_all    = np.concatenate(P_x_sim_list)
    Pdot_all = np.concatenate(Pdot_x_sim_list)
    S_all    = np.concatenate(S_x_rcs_abs_list)

    # ============================
    #     CUMULATIVE HISTOGRAMS
    # ============================

    # --- Period ---
    axP.hist(
        P_all[P_all > 0],
        bins=np.logspace(-2, 3, 30),
        histtype="step",
        cumulative=True,
        density=True,
        color=colors[j],
        lw=2,
        label=B0_labels[j]
    )

    # --- Pdot ---
    axPdot.hist(
        Pdot_all[Pdot_all > 0],
        bins=np.logspace(-12, -8, 30),
        histtype="step",
        cumulative=True,
        density=True,
        color=colors[j],
        lw=2,
        label=B0_labels[j]
    )

    # --- Flux ---
    axS.hist(
        S_all[S_all > 0],
        bins=np.logspace(-14, -9, 30),
        histtype="step",
        cumulative=True,
        density=True,
        color=colors[j],
        lw=2,
        label=B0_labels[j]
    )



# ========================================
#        OBSERVED CUMULATIVE CURVES
# ========================================

axP.hist(
    P_x_obs_young,
    bins=np.logspace(-2, 3, 30),
    histtype="step",
    cumulative=True,
    density=True,
    color="dimgrey",
    lw=4,
    label="Observed"
)

axPdot.hist(
    Pdot_x_obs_young,
    bins=np.logspace(-12, -8, 30),
    histtype="step",
    cumulative=True,
    density=True,
    color="dimgrey",
    lw=4,
    label="Observed"
)

axS.hist(
    S_x_abs_obs_young,
    bins=np.logspace(-14, -9, 30),
    histtype="step",
    cumulative=True,
    density=True,
    color="dimgrey",
    lw=4,
    label="Observed"
)

# ============================
#        AXIS FORMATTING
# ============================

for ax in (axP, axPdot, axS):
    ax.tick_params(axis="y", right=False,left=False)
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.tick_params(axis="both",labelsize=35)
    #ax.set_yticks([])
    ax.minorticks_off()
    ax.tick_params(top=False, bottom=True)
    
axP.tick_params(axis="y", left=True)
axPdot.tick_params(labelleft=False)
axS.tick_params(labelleft=False)
axPdot.set_xticks([1e-11,1e-10,1e-9])
axP.set_xticks([ 1, 10])
axS.set_xticks([1e-13,1e-11,1e-9])

axP.set_xlabel("P [s]")
axPdot.set_xlabel(r"$\dot{P}$ [s/s]")
axS.set_xlabel(r"$S_{X,\rm abs}$ [erg s$^{-1}$ cm$^{-2}$]")

handles, labels = axP.get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.97, 0.6),
    frameon=False,
    fontsize=22
)
axP.set_xlim(1e-1,1e2)
axPdot.set_xlim(1e-12,1e-8)
axS.set_xlim(1e-14,1e-9)



axP.set_ylim(0,1)
axPdot.set_ylim(0,1)
axS.set_ylim(0,1)

plt.savefig(
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/plots/"
    "cumulative_magnetars.pdf",
    bbox_inches="tight"
)
#plt.tight_layout()
plt.show()


# KS test for simulated vs observed magnetar

The null hiypothesis in this test is that both distributions are identical, therefore the statistics KS is the maximun distance between the two distributions. Therefore, for p_values > 0.05 you have a 95% confidence that the thre is no significant evidence that they differ. Otherwise, if p_values < 0.05 the distributions are statistically different.

In [ ]:
print("\n============================")
print(" K–S TEST RESULTS")
print("============================\n")

ks_results = {
    "B0": [],
    "KS_P": [], "p_P": [], "valid_P": [],
    "KS_Pdot": [], "p_Pdot": [], "valid_Pdot": [],
    "KS_S": [], "p_S": [], "valid_S": []
}

def validity(p):
    return "VALID (p > 0.01)" if p > 0.01 else "NOT VALID (p ≤ 0.01)"

for j, exp_name in enumerate(exp_folders):
    base_path = parent_path / exp_name
    
    S_x_rcs_abs_list, P_x_sim_list, Pdot_x_sim_list = [], [], []
    
    # -------- FIND ALL SIMULATION DIRECTORIES ----------
    sim_dirs = sorted(
        [d for d in base_path.iterdir() 
         if d.is_dir() and d.name.startswith("simulation_")],
        key=lambda p: int(p.name.split("_")[1])  # numeric sort
    )
    
    # -------- LOAD SIMULATIONS --------
    for sim_dir in sim_dirs:
        sim_path = sim_dir / "final_population.pkl.gz"
        if not sim_path.exists():
            continue  # skip missing files

        data_pop = pd.read_pickle(sim_path, compression="gzip")
        
        magnetars = data_pop[
            (np.log10(data_pop[('B','[G]')]) >= 13.5) &
            (data_pop[('S_x_rcs_abs','[erg s^-1 cm^-2]')] > 1e-14)
        ]
        
        S_x_rcs_abs_list.append(magnetars[('S_x_rcs_abs','[erg s^-1 cm^-2]')].to_numpy())
        P_x_sim_list.append(magnetars[('P','[s]')].to_numpy())
        Pdot_x_sim_list.append(magnetars[('P_dot','[s s^-1]')].to_numpy())

    # -------- MERGE --------
    P_all    = np.concatenate(P_x_sim_list)
    Pdot_all = np.concatenate(Pdot_x_sim_list)
    S_all    = np.concatenate(S_x_rcs_abs_list)

    # -------- REMOVE INVALID VALUES --------
    P_all    = P_all[P_all > 0]
    Pdot_all = Pdot_all[Pdot_all > 0]
    S_all    = S_all[S_all > 0]

    # ===== KS TEST =====
    ksP,  pP  = ks_2samp(P_all,    P_x_obs_young)
    ksPd, pPd = ks_2samp(Pdot_all, Pdot_x_obs_young)
    ksS,  pS  = ks_2samp(S_all,    S_x_abs_obs_young)
    
    # -------- STORE RESULTS --------
    ks_results["B0"].append(B0_labels[j])
    ks_results["KS_P"].append(ksP)
    ks_results["p_P"].append(pP)
    ks_results["valid_P"].append(validity(pP))

    ks_results["KS_Pdot"].append(ksPd)
    ks_results["p_Pdot"].append(pPd)
    ks_results["valid_Pdot"].append(validity(pPd))

    ks_results["KS_S"].append(ksS)
    ks_results["p_S"].append(pS)
    ks_results["valid_S"].append(validity(pS))

    # -------- PRINT RESULTS --------
    print(f"==== B0 = {B0_labels[j]} ====")
    print(f"  Period      : KS = {ksP: .3f},  p = {pP:.3e}  → {validity(pP)}")
    print(f"  Pdot        : KS = {ksPd: .3f}, p = {pPd:.3e}  → {validity(pPd)}")
    print(f"  Flux        : KS = {ksS: .3f},  p = {pS:.3e}  → {validity(pS)}\n")


# XDINS observations vs simulations

In [ ]:
def collect_xdins_data(parent_path, exp_folders, n_simulations=None):
    """
    Returns:
        data_by_exp[exp_name] = {
            "P":    np.array,
            "Pdot": np.array,
            "S":    np.array,
        }
    """

    data_by_exp = {}

    for exp_name in exp_folders:
        base_path = parent_path / exp_name

        P_list, Pdot_list, S_list = [], [], []

        # ---------------------------------------------
        # Find all simulation_* folders (unordered OK)
        # ---------------------------------------------
        sim_dirs = [d for d in base_path.iterdir() 
                    if d.is_dir() and d.name.startswith("simulation_")]

        # Sort numerically by the integer after "simulation_"
        sim_dirs = sorted(sim_dirs, key=lambda p: int(p.name.split("_")[1]))

        # ---------------------------------------------
        # Load each simulation folder safely
        # ---------------------------------------------
        for sim_dir in sim_dirs:

            sim_path = sim_dir / "final_population.pkl.gz"
            if not sim_path.exists():
                continue  # skip missing

            try:
                data_pop_x = pd.read_pickle(sim_path, compression="gzip")
            except Exception as e:
                print(f"Warning: could not load {sim_path}: {e}")
                continue

            # --- geometry ---
            r   = data_pop_x["r"]["[kpc]"].to_numpy()
            phi = data_pop_x["phi"]["[rad]"].to_numpy()
            z   = data_pop_x["z"]["[kpc]"].to_numpy()

            x = r * np.cos(phi)
            y = r * np.sin(phi)
            dist_from_sun = np.sqrt(
                (x - 0.0)**2 +
                (y - 8.3)**2 +
                (z - 0.006)**2
            )

            # spatial & flux masks
            mask_dist = dist_from_sun <= 0.5
            mask_flux = data_pop_x[('S_x_rcs_abs', '[erg s^-1 cm^-2]')] > 5e-13

            data_pop_x = data_pop_x[mask_dist & mask_flux]

            # age filtering
            mask_age_real = data_pop_x["age"]["[yr]"] > 1e5
            xdins = data_pop_x[mask_age_real]

            # store arrays (skip empty automatically)
            S_list.append(xdins[('S_x_rcs_abs','[erg s^-1 cm^-2]')].to_numpy())
            P_list.append(xdins[('P','[s]')].to_numpy())
            Pdot_list.append(xdins[('P_dot','[s s^-1]')].to_numpy())

        # ------------------------------------------------
        # Concatenate all simulations for this experiment
        # ------------------------------------------------
        P_all    = np.concatenate(P_list)    if P_list    else np.array([])
        Pdot_all = np.concatenate(Pdot_list) if Pdot_list else np.array([])
        S_all    = np.concatenate(S_list)    if S_list    else np.array([])

        data_by_exp[exp_name] = {
            "P": P_all,
            "Pdot": Pdot_all,
            "S": S_all,
        }

    return data_by_exp


xdins_exp_folders = [
    "exp_7_5e13",
    "exp_1e14",
     "exp_1_5e14",
     "exp_2e14",
    "exp_2_5e14",
    "exp_5e14",
    "exp_7_5e14",
    "exp_1e15",
]
# ---- call once ----
parent_path = Path("/Users/celsapardoaraujo/Documents/data/magnetar-br/simulations_sim_vs_obs_figure/xdins")
data_by_exp = collect_xdins_data(parent_path, xdins_exp_folders, 10)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey = True)
plt.subplots_adjust(wspace=0)

axP, axPdot, axS = axes

# Yellow → Green gradient for increasing field
cmap = cm.get_cmap("YlGn")
colors = cmap(np.linspace(0.2, 0.9, len(xdins_exp_folders)))

# ============================
#     CUMULATIVE HISTOGRAMS
# ============================
for j, exp_name in enumerate(xdins_exp_folders):
    d = data_by_exp[exp_name]
    P_all    = d["P"]
    Pdot_all = d["Pdot"]
    S_all    = d["S"]

    axP.hist(
        P_all[P_all > 0],
        bins=np.logspace(-2, 3, 30),
        histtype="step",
        cumulative=True,
        density=True,
        color=colors[j],
        lw=2,
        label=B0_labels[j]
    )

    axPdot.hist(
        Pdot_all[Pdot_all > 0],
        bins=np.logspace(-15, -8, 30),
        histtype="step",
        cumulative=True,
        density=True,
        color=colors[j],
        lw=2,
        label=B0_labels[j]
    )

    axS.hist(
        S_all[S_all > 0],
        bins=np.logspace(-14, -9, 30),
        histtype="step",
        cumulative=True,
        density=True,
        color=colors[j],
        lw=2,
        label=B0_labels[j]
    )

# ========================================
#        OBSERVED CUMULATIVE CURVES
# ========================================

axP.hist(
    P_x_obs_xdins,
    bins=np.logspace(-2, 3, 30),
    histtype="step",
    cumulative=True,
    density=True,
    color="dimgrey",
    lw=4,
    label="Observed"
)

axPdot.hist(
    Pdot_x_obs_xdins,
    bins=np.logspace(-15, -8, 30),
    histtype="step",
    cumulative=True,
    density=True,
    color="dimgrey",
    lw=4,
    label="Observed"
)

axS.hist(
    S_x_abs_obs_xdins,
    bins=np.logspace(-14, -9, 30),
    histtype="step",
    cumulative=True,
    density=True,
    color="dimgrey",
    lw=4,
    label="Observed"
)

# ============================
#        AXIS FORMATTING
# ============================

for ax in (axP, axPdot, axS):
    # ✅ Remove ALL y-ticks and y-tick labels from every panel
    #ax.set_yticks([])
    ax.tick_params(axis="y", left=False,right=False)
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.tick_params(axis="both",labelsize=35)
    ax.minorticks_off()
    ax.tick_params(top=False, bottom=True)

# ✅ Hide y tick labels only on the right panels (KEEP THEM ON THE LEFT)
axP.tick_params(axis="y", left=True)
axPdot.tick_params(labelleft=False)
axS.tick_params(labelleft=False)
axPdot.set_xticks([1e-15, 1e-12, 1e-9])
axP.set_xticks([ 1, 10, 1e2])
axS.set_xticks([1e-13,  1e-11, 1e-9])

axP.set_xlabel("P [s]")
axPdot.set_xlabel(r"$\dot{P}$ [s/s]")
axS.set_xlabel(r"$S_{X,\rm abs}$ [erg s$^{-1}$ cm$^{-2}$]")

handles, labels = axP.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.96, 0.6),
    frameon=False,
    fontsize=22
)

axP.set_xlim(1e-1, 1e3)
axPdot.set_xlim(1e-16, 1e-8)
axS.set_xlim(1e-14, 1e-9)

axP.set_ylim(0, 1)
axPdot.set_ylim(0, 1)
axS.set_ylim(0, 1)



#plt.tight_layout()
plt.savefig(
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/plots/"
    "cumulative_xdins.pdf",
    bbox_inches="tight"
)
plt.show()


In [ ]:
from scipy.stats import ks_2samp
import numpy as np

print("\n============================")
print(" K–S TEST RESULTS")
print("============================\n")

ks_results = {
    "B0": [],
    "KS_P": [], "p_P": [], "valid_P": [],
    "KS_Pdot": [], "p_Pdot": [], "valid_Pdot": [],
    "KS_S": [], "p_S": [], "valid_S": []
}

def validity(p):
    return "VALID (p > 0.01)" if p > 0.01 else "NOT VALID (p ≤ 0.01)"
# remove invalid observed values once (important!)
P_obs    = P_x_obs_xdins[P_x_obs_xdins > 0]
Pdot_obs = Pdot_x_obs_xdins[Pdot_x_obs_xdins > 0]
S_obs    = S_x_abs_obs_xdins[S_x_abs_obs_xdins > 0]

for j, exp_name in enumerate(xdins_exp_folders):

    sim = data_by_exp[exp_name]

    # remove invalid simulated values
    P_all    = sim["P"][sim["P"] > 0]
    Pdot_all = sim["Pdot"][sim["Pdot"] > 0]
    S_all    = sim["S"][sim["S"] > 0]

    # ===== KS TESTS =====
    ksP,  pP  = ks_2samp(P_all,    P_obs)
    ksPd, pPd = ks_2samp(Pdot_all, Pdot_obs)
    ksS,  pS  = ks_2samp(S_all,    S_obs)

    # store results
    ks_results["B0"].append(B0_labels[j])

    ks_results["KS_P"].append(ksP)
    ks_results["p_P"].append(pP)
    ks_results["valid_P"].append(validity(pP))

    ks_results["KS_Pdot"].append(ksPd)
    ks_results["p_Pdot"].append(pPd)
    ks_results["valid_Pdot"].append(validity(pPd))

    ks_results["KS_S"].append(ksS)
    ks_results["p_S"].append(pS)
    ks_results["valid_S"].append(validity(pS))

    # print output (same format as yours)
    print(f"==== B0 = {B0_labels[j]} ====")
    print(f"  Period : KS = {ksP: .3f},  p = {pP:.3e}  → {validity(pP)}")
    print(f"  Pdot   : KS = {ksPd: .3f}, p = {pPd:.3e}  → {validity(pPd)}")
    print(f"  Flux   : KS = {ksS: .3f},  p = {pS:.3e}  → {validity(pS)}\n")
